# Proyecto de Primer Bimestre  
## Sistema de Recuperación de Información  

**Integrantes:** Bautista Alexis - Correa Francisco  
**Fecha de entrega:** 1 de junio de 2026

### a. Construcción del índice

In [ ]:
import preprocesamiento

Leer un corpus de documentos en texto plano.

In [3]:
corpus = preprocesamiento.cargar_corpus()

Se encontraron 8 archivos CSV en el directorio.
Carga completa. El corpus tiene un total de 51081 documentos individuales.


Procesamiento básico: tokenización, normalización y remoción de stopwords.

In [4]:
corpus_procesado = []
for doc in corpus:
    tokens_doc = preprocesamiento.preprocesar(doc)
    corpus_procesado.append(tokens_doc)

print(f"Se han preprocesado {len(corpus_procesado)} documentos.")

Se han preprocesado 51081 documentos.


In [4]:
#Verificacion de que se proceso
# print(corpus_procesado)

Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia.

In [5]:
import indice_invertido

In [6]:
indice = indice_invertido.crear_indice_invertido(corpus_procesado)

El indice a sido creado correctamente


In [7]:
# prueba rapida
if "bank" in indice:
    for doc_id, freq in list(indice["bank"].items())[:5]:
        print(f"{doc_id}: {freq}")
else:
    print("El término no existe en el corpus")

4: 1
9: 15
12: 5
13: 2
14: 3


**Ejemplos de interpretacion:**

4: 1 -> En el documento con el ID 4, la palabra "bank" aparece 1 vez.  
9: 15 -> En el documento con el ID 9, la palabra "bank" se repite 15 veces.  
12: 5 -> En el documento con el ID 12, la palabra "bank" se encuentra 5 veces.

In [7]:
# prueba rapida
print(indice.get("DDada", "El término no existe en el corpus"))

El término no existe en el corpus


### b. Modelo de recuperación

Implementar recuperación basada en similitud Jaccard utilizando vectores binarios

In [8]:
import modelos

In [9]:
query = input ("Ingrese una consulta: ")

In [10]:
ranking = modelos.recuperar_jaccard(query, corpus_procesado)

print(f"Resultados para: '{query}'")
for doc_id, score in ranking[:5]: # Mostrar el top 5
    
    print(f"Documento ID: {doc_id} | Similitud: {score * 100:.2f}%")

Resultados para: 'cat in house'
Documento ID: 1431 | Similitud: 6.25%
Documento ID: 11107 | Similitud: 6.25%
Documento ID: 24665 | Similitud: 6.25%
Documento ID: 29075 | Similitud: 6.25%
Documento ID: 34561 | Similitud: 6.25%


Implementar recuperación basada en similitud de coseno utilizando TF-IDF

In [11]:
ranking_tfidf = modelos.recuperar_tfidf(query, corpus_procesado)

print(f"Resultados TF-IDF para: '{query}'")
for doc_id, score in ranking_tfidf[:5]: # Mostrar el top 5
    # similitud coseno entre 0 y 1
    print(f"Documento ID: {doc_id} | Similitud Coseno: {score:.4f}")

Resultados TF-IDF para: 'cat in house'
Documento ID: 10521 | Similitud Coseno: 0.1940
Documento ID: 23996 | Similitud Coseno: 0.1940
Documento ID: 48534 | Similitud Coseno: 0.1940
Documento ID: 5509 | Similitud Coseno: 0.1441
Documento ID: 16898 | Similitud Coseno: 0.1441


Implementar recuperación con BM25.

In [15]:
ranking_bm25 = modelos.recuperar_bm25(query, corpus_procesado, indice)

print(f"Resultados BM25 para: '{query}'")
for doc_id, score in ranking_bm25[:5]:
    print(f"Documento ID: {doc_id} | Score BM25: {score:.4f}")

Resultados BM25 para: 'cat in house'
Documento ID: 29091 | Score BM25: 7.1117
Documento ID: 34577 | Score BM25: 7.1117
Documento ID: 10521 | Score BM25: 6.2963
Documento ID: 23996 | Score BM25: 6.2963
Documento ID: 48534 | Score BM25: 6.2963


**Interpretación de Métricas BM25**

* **Scores no normalizados:** Los valores obtenidos (ej. 7.1117) no son porcentajes ni probabilidades (0-100%). Son métricas relativas que solo sirven para comparar qué documento es más relevante que otro dentro de la *misma* consulta.
* **Empates matemáticos:** Los scores idénticos ocurren cuando los documentos son textos duplicados, o cuando coinciden exactamente en su longitud total y en la cantidad de veces que repiten los términos buscados.
* **Criterios de relevancia:** Un score alto indica que el documento contiene las palabras más "raras" de la búsqueda (alto IDF), las menciona de forma natural sin hacer spam (saturación de TF) y es un texto relativamente conciso (penalización a documentos muy largos).

### c. Interfaz básica

### d. Recuperación semántica con embeddings

• Generar embeddings para los documentos del corpus utilizando un modelo preentrenado.  
• Generar embeddings para las consultas de texto libre.  
• Almacenar los embeddings en una base de datos vectorial, como ChromaDB o FAISS.  
• Recuperar los documentos más similares usando búsqueda vectorial.  
• Mostrar un ranking de resultados basado en similitud vectorial.  

Para modelos clasicos como TF-IDF, BM25 es necesrio tener los tokens limpios y el stemming (ej. ["japan", "bank", "tax"]). Sin embargo, a los modelos semánticos (Transformers) les hace daño el preprocesamiento agresivo. Estos modelos necesitan leer el texto con su sintaxis, puntuación y conectores (stop words) para entender el contexto real de la oración.

Por esto se usara el corpus original para esta seccion. Ademas se decidio usar la base de datos vectorial FAISS. Por ultimo se decidio usar el modelo all-MiniLM-L6-v2 ya que es el estándar de la industria para este tipo de proyectos académicos porque es extremadamente rápido, pesa poco y ofrece una precisión altísima para representar oraciones en inglés.

In [16]:
import modelo_semantico

modelo_transformer, base_vectorial_faiss = modelo_semantico.construir_indice_faiss(corpus)

c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando el modelo preentrenado 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3076.23it/s]


Generando embeddings para 51081 documentos...
(Esto puede tomar unos minutos dependiendo del procesador)


Batches: 100%|██████████| 1597/1597 [19:42<00:00,  1.35it/s]


Construyendo el índice FAISS (Dimensión: 384)...
Índice FAISS completado con 51081 vectores.


In [20]:
#prueba
mi_busqueda = "cat in blue house house"

ranking_semantico = modelo_semantico.recuperar_semantico(
    query_texto=mi_busqueda, 
    modelo=modelo_transformer, 
    indice_faiss=base_vectorial_faiss, 
    top_k=5
)

print(f"Resultados Semánticos para: '{mi_busqueda}'")
for doc_id, score in ranking_semantico:
    print(f"Documento ID: {doc_id} | Similitud Semántica: {score:.4f}")

Resultados Semánticos para: 'cat in blue house house'
Documento ID: 26401 | Similitud Semántica: 0.3986
Documento ID: 50174 | Similitud Semántica: 0.3986
Documento ID: 28954 | Similitud Semántica: 0.3932
Documento ID: 34440 | Similitud Semántica: 0.3932
Documento ID: 5376 | Similitud Semántica: 0.3931
